# ISYE 524 Project: 
# <center> **Optimal Reverse Supply Chain**: </center>
## <center> Modeling Multi-Echelon, Multi-Indenture Systems </center>
---
## Professor: Amanda Smith
## Project Members: Seth Ockerman and Mitchell Stachowiak
### University of Wisconsin - Madison
### Spring 2026
---
# <center><u> Report </u></center>

---
# <center>Introduction</center>

[The first few sentences should give a quick overview of the entire project. Then, elaborate with a description of the problem that will be solved, a brief history (with citations, if appropriate) of how the problem came about, why it’s important/interesting, and any other interesting facts you’d like to talk about. You should address and explain where the problem data is coming from (research? the internet? synthetically generated?) Also give an outline of the rest of the report. Include any assumptions you are making to simplify the problem. This section should be accessible to someone that has not taken this course.]


The project seeks to model the problem of optimally fulfilling demands of a **reverse supply** chain which is **multi-echelon** and **multi-indenture** by using the tooling of linear and integer optimization.

The project was inspired by the work of Craig C. Sherbrooke, in particular his work in developing marginal analysis tools like the METRIC model for the USAF as well as his textbook *Optimal Inventory Modeling of Systems: Multi-Echelon Techniques*, where we desire to attempt to implement our models using LP methods rather than his marginal approach; moreover, while his work conserved defense applications, the problems faced in the defense industry are readily mirrored in the commercial sector such as commercial aviation or trucking where the same low demand and high cost assumptions can be applied. In these areas, it is of the utmost importance to prevent misallocation of demands and resources as turnarounds are usually slow, mistakes are costly, and inefficiencies can easily compound into other sectors like commerce, travel, and defense.

To make our model concrete, we will be considering the motivating example of repair logistics; specifically, we will be considering how to optimally repair components of aircraft. Given the relative sparsity of aircraft relative to that of automobiles, some components are produced to order with relatively small inventory buffer; moreover, the manufacture of such equipment is prohibitively expensive and time consuming to constantly order new components for repairs; thus, repair logistics is employed to minimize cost of maintenance and down-time of aircraft.

A **reverse supply-chain** is a supply-chain in which customers generate demand for repair or refurbishment services rather than consumption of new products. In our motivating example, failed components are returned, diagnosed, and repaired through a network rather than purchasing new components like in a traditional supply-chain.

A **multi-echelon** system consists of multiple hierarchical levels, called echelons, through which items must flow. In our motivating example, the highest echelon might be some central depot(s), followed by regional warehouses, local repair shops, and potentially specialized technicians. These echelons are connected via a logistics network that can be represented as a directed graph with costs associated to transportation, time delays, and/or capacity limits.

A **multi-indenture** system consists of multiple hierarchical levels, called indentures, which capture the fact that repairing a component may require the availability of subcomponents. In our motivating example, repairing a Line Replaceable Unit (LRU), which may be the demanded reparable, may require multiple Shop Replaceable Units (SRUs), which may be specific components allocated at the shop echelon, each of which may themselves require further subcomponents.

In the combination of multi-echelon and multi-indenture, the system includes both flow decisions across the network and repair dependencies across indenture levels which further constraint repair across the network by component availability. The goal of the model is to determine how to route and allocate LRUs, and SRUs, across the network so that repair demands are fulfilled efficiently, ideally at the lowest feasible echelon, while respecting network constraints and component dependencies while maximizing some designed objective/cost function.

In practice the **objective/cost function** would be curated by the specific customer to meet their desired criteria. In general, the function would be some parameterized metric over a possible combination of cost, time, and performance; specifically, it would be designed such that we minimize the total cost and time to meet the demands while maximizing some performance metric, or minimize total cost and time under the additional constraint of the performance metric. The performance metric tracks some important desired quantity reflecting how operational impact for the customer. In our motivating example, such a metric could track the percentage of aircraft availability with repairs or some measure of overall ability of the customer to meet demands with specific aircraft availability with repairs.

Note that the performance metric can be incredibly useful as a constraint or objective as the repair of some components can be significantly more operationally important than others. In our motivating example, the repair of some group of parts, which may take more time and money, may allow a downed aircraft to fly while other repairs, which would otherwise be prioritized by cost/time, may not and so is far less valuable to the customer. In defense applications, this performance metric is often number of potential sorties available which is critical to national security and readiness.

---
# <center>Discussion</center>

[An explanation of the work done so far and what work remains before the project is completed. To the best of your ability, give rough estimates of how long you expect the remaining steps to take.]

---
# <center>Issues/Concerns</center>

[Briefly summarize any major issues or concerns you have about your project at this point. Do you anticipate being able to solve these issues?]

---
# <center>Mathematical Model</center>

[Explain the decision variables, the constraints, and the objective function. Finally, show the optimization problem written in standard form or, if standard form makes your model more difficult to understand, some other "clean" model form that is easy to interpret.  Equations should be formatted in LaTeX using correct math notation within the IJulia notebook. This section should be accessible to someone that has taken this course.]

---
# <center>Julia JuMP-Gurobi Implementation</center>

[Here, you should code up your model in Julia + JuMP and solve it. Your code should be clean, easy to read, well annotated and commented, and it should compile! I suggest having multiple code blocks separated by text blocks that explain the various parts of your solution. You may also solve several versions of your problem with different models/assumptions. Make sure that anything you include in your code output is discussed/explained and integrated into the report. All other output should be suppressed.]

---
# <center>Results</center>

[Here, you display and discuss the results. Show figures, tables, plots, images, trade-off curves, or whatever else you can think of to best illustrate your results. The discussion should explain what the results mean, and how to interpret them. You should also explain the limitations of your approach/model and how sensitive your results are to the assumptions you made.]

---
# <center>Conclusion</center>

[Summarize your findings and your results, and talk about at least one possible future direction; something that might be interesting to pursue as a follow-up to your project. Be specific enough with your follow-up idea to show that you have given it some thought and that you think it’s actually doable!]

In [2]:
import Pkg
Pkg.add("Gurobi", io=devnull)
Pkg.add("DataFrames", io=devnull)
Pkg.add("CSV", io=devnull)
Pkg.add("NamedArrays", io=devnull)
Pkg.add("Distributions", io=devnull)

In [3]:
# Mitchell Gurobi Location Desktop
# ENV["GUROBI_HOME"] = "C:\\Program Files\\gurobi1100\\win64"
# Mitchell Gurobi Location Laptop
ENV["GUROBI_HOME"] = "C:\\Program Files\\gurobi1301\\win64"
Pkg.build("Gurobi", io=devnull)

In [4]:
using JuMP
using Gurobi
using Random
using Distributions
using DataFrames
using CSV
using NamedArrays

In [21]:
# Load Data
node_df = CSV.read("level0data/nodes.csv",DataFrame,delim=',')
arc_df = CSV.read("level0data/arcs.csv",DataFrame,delim=',')
bom_df = CSV.read("level0data/bom.csv",DataFrame,delim=',')
demand_df = CSV.read("level0data/demand.csv",DataFrame,delim=',')
node_inv_df = CSV.read("level0data/node_inventory.csv",DataFrame,delim=',')

# Convert node_df to Node Information
#--------------------------------------------------
struct Node
    name::String
    node_type::String
    loc::Tuple{Float64, Float64}
end
node_dict = Dict() # dictionary of nodes -> Node struct
for df_node in eachrow(node_df)
    node_id = df_node.node_id
    node_type = df_node.node_type
    (x, y) = (df_node.x, df_node.y)
    node = Node(node_id, node_type, (x, y))
    node_dict[node_id] = node
end
nodes = collect(keys(node_dict))         # list of nodes 


# Convert arc_df to Arc Information
#--------------------------------------------------
struct Arc
    name::String
    fromto::Tuple{String, String}
    trans_cost::Float64
end
arc_dict = Dict() # dictionary of arcs -> Arc struct
for df_arc in eachrow(arc_df)
    arc_id = df_arc.arc_id
    node_from = df_arc.from_node
    node_to = df_arc.to_node
    trans_cost = df_arc.cost
    arc = Arc(arc_id, (node_from, node_to), trans_cost) # add arc to edges list
    arc_dict[(node_from, node_to)] = arc # add Arc to arc_dict
end
arcs = collect(keys(arc_dict))         # list of arcs of network
incoming = Dict()
outgoing = Dict()
for (node_from, node_to) in arcs
    push!(get!(incoming, node_to, []), node_from)
    push!(get!(outgoing, node_from, []), node_to)
end


# Convert bom_df to Bill of Material (BOM) Dictionary
#--------------------------------------------------
struct Component
    comp::String
    comp_type::String
    repair_cost::Float64
    subcomps::Vector{Tuple{String, Int}}
end
BOM = Dict()
for bom_item in eachrow(bom_df)
    comp_name = bom_item.parent_component_id
    comp_type = bom_item.parent_component_class
    comp_repair_cost = bom_item.parent_component_cost
    comp_repair_list = []
    sub_bom = subset(bom_df, :parent_component_id => ByRow(==(comp_name)))
    for sub_comp in eachrow(sub_bom)
        sub_comp_name = sub_comp.child_component_id
        sub_comp_num = sub_comp.quantity_required
        push!(comp_repair_list, (sub_comp_name, sub_comp_num))
    end
    comp = Component(comp_name, comp_type, comp_repair_cost, comp_repair_list)
    BOM[comp_name] = comp
end
components_wo_parts = collect(keys(BOM))
# add parts components to BOM
part_bom = subset(bom_df, :child_component_class => ByRow(==("part")))
for bom_item in eachrow(bom_df)
    comp_name = bom_item.child_component_id
    comp_type = "part"
    comp_repair_cost = 0
    comp_repair_list = []
    comp = Component(comp_name, comp_type, comp_repair_cost, comp_repair_list)
    BOM[comp_name] = comp
end
components = collect(keys(BOM))
part_components = setdiff(components, components_wo_parts)
subcomp_dict = Dict()
for comp in components
    subcomp_bom = subset(bom_df, :child_component_id => ByRow(==(comp)))
    subcomp_for_lst = []
    for subcomp_for in eachrow(subcomp_bom)
        push!(subcomp_for_lst, (subcomp_for.parent_component_id, subcomp_for.quantity_required))
    end
    subcomp_dict[comp] = subcomp_for_lst
end


# Convert demand_df to Demand Information
#--------------------------------------------------
struct Demand
    name::String
    loc::String
    component::String
    num::Int
end
demands = Dict()
for df_demand in eachrow(demand_df)
    demand_id = df_demand.demand_id
    demand_node = df_demand.node_id
    demand_comp = df_demand.component_id
    quantity = df_demand.quantity
    demand = Demand(demand_id, demand_node, demand_comp, quantity)
    demands[(demand_node, demand_comp)] = demand
end


# Convert node_inv_df to Node Inventory Information
#--------------------------------------------------
struct Stock
    loc::String
    component::String
    num::Int
end
inventory = Dict()
for df_stock in eachrow(node_inv_df)
    stock_node = df_stock.node_id
    stock_comp = df_stock.component_id
    quantity = df_stock.quantity
    stock = Stock(stock_node, stock_comp, quantity)
    inventory[(stock_node, stock_comp)] = stock
end

In [24]:
# Model
m = Model(Gurobi.Optimizer)

# Variables
@variable(m, x[arc in arcs, comp in components] >= 0, Int)   # flow of x comp through arc
@variable(m, r[node in nodes, comp in components] >= 0, Int) # r repairs of comp at node

# Constraints
# Flow conservation
for comp in components, node in nodes
    inflow  = sum(x[(node_from, node), comp] for node_from in incoming[node])
    outflow = sum(x[(node, node_to), comp] for node_to in outgoing[node])
    demand = get(demands, (node, comp), Demand("", "", "", 0)).num
    stock = get(inventory, (node, comp), Stock("", "", 0)).num
    repairs = r[node, comp]
    consumed = sum(num * r[node, comp_repaired] for (comp_repaired, num) in subcomp_dict[comp]; init=0)
    @constraint(m, stock + inflow + repairs >= outflow + consumed + demand) # must have net positive comp at node
        # otherwise means that every component must be used everywhere. Check if allows for illegal creation somewhere.
        # I don't think so as only inflow, outflow, and repair creation and consumtion variable but rpeair should be fine by construction.
end
# No free creation of part components
for comp in part_components, node in nodes
    @constraint(m, r[node, comp] == 0)
end

#=
# Multi-indenture (BOM constraints)
# If repairing parent k, must consume subcomponents
for comp in components
    req = BOM[(comp, subcomp)]
    for node in nodes
        @constraint(m, r[node, comp] * req <= r[node, subcomp])   # parts repaired at shop j must have necessary components available
    end
end
=#


# Objective
@objective(m, Min,
           sum(arc_dict[arc].trans_cost * x[arc, comp] for comp in components, arc in arcs) + # transport cost
           sum(BOM[comp].repair_cost * r[node, comp] for comp in components, node in nodes)   # repair cost
           )


# Solve
optimize!(m)

# Output
println("Objective value: ", objective_value(m))

for comp in components, (node_from, node_to) in arcs
    val = value(x[(node_from, node_to), comp])
    if val > 1e-3
        println("Flow of $comp: $node_from -> $node_to = $val")
    end
end
for comp in components, node in nodes
    val = value(r[node, comp])
    if val > 1e-3
        println("Repair component $comp at node $node: $val")
    end
end

Set parameter Username
Set parameter LicenseID to value 2807856
Academic license - for non-commercial use only - expires 2027-04-15
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 645 rows, 1827 columns and 4029 nonzeros (Min)
Model fingerprint: 0x5adb56ec
Model has 1467 linear objective coefficients
Variable types: 0 continuous, 1827 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [1e+01, 1e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+02]
Found heuristic solution: objective 23728.000000
Presolve removed 213 rows and 216 columns
Presolve time: 0.00s
Presolved: 432 rows, 1611 columns, 3597 nonzeros
Variable types: 0 continuous, 1611 integer (0 binary)

Root relaxation: objective 1.9136

---
# <center>Citations</center>

Citations